# **Lab2: Text Classification using LSTM**
This Colab notebook demonstrates how to build and train a sentiment classification model using an LSTM (Long Short-Term Memory) network on the IMDB movie reviews dataset. The objective is to classify each review as either positive or negative. The notebook walks through key steps such as data loading, preprocessing, building an embedding layer followed by an LSTM for sequence learning, and evaluating the model’s performance. Through this hands-on exercise, learners will understand how LSTM networks capture word dependencies to improve binary sentiment prediction.

# 1. Imports and Setup

- Imports: Import TensorFlow and necessary modules from Keras for building the LSTM model.
- Parameters: Set parameters such as max_features (maximum number of words to consider as features in the vocabulary), maxlen (maximum length of sequences), batch_size (number of samples per gradient update), embedding_dims (dimensionality of the dense embedding), and epochs (number of training iterations).

In [8]:
# Importing necessary libraries
import tensorflow as tf  # Import TensorFlow library for deep learning operations
from tensorflow.keras.datasets import imdb  # Import IMDB dataset for sentiment analysis
from tensorflow.keras.preprocessing.sequence import pad_sequences  # Utility for padding sequences
from tensorflow.keras.models import Sequential  # Sequential model type for layer stacking
from tensorflow.keras.layers import LSTM, Dense, Embedding  # Different types of neural network layers
from keras.utils import plot_model

# Setting parameters for the model and training
max_features = 20000  # Number of words to consider as features from the dataset
maxlen = 200  # Cuts off texts after this number of words
batch_size = 32  # Number of samples per gradient update during training
embedding_dims = 128  # Dimensionality of the dense embedding space
epochs = 5  # Number of training iterations over the entire dataset

# 2. Loading Data

Load the IMDb movie reviews dataset using imdb.load_data(). num_words=max_features limits the vocabulary to the top max_features most frequent words.

In [9]:
# Load the IMDb dataset
print('Loading data...')
(x_train_full, y_train_full), (x_test_full, y_test_full) = imdb.load_data(num_words=max_features)

# Selecting a subset of the data for computational efficiency
subset_size = 2000
x_train = x_train_full[:subset_size]  # Subset of training data
y_train = y_train_full[:subset_size]  # Corresponding subset of training labels
x_test = x_test_full[:subset_size]    # Subset of test data
y_test = y_test_full[:subset_size]    # Corresponding subset of test labels

# Output information about the loaded data
print(len(x_train), 'train sequences')  # Number of training sequences
print(len(x_test), 'test sequences')    # Number of test sequences


Loading data...
2000 train sequences
2000 test sequences


The dataset is encoded by default. The below code allows us to see the original sentences:

In [10]:
# Get the word-to-index mapping from the dataset
word_to_index = imdb.get_word_index()
word_to_index = {k:(v+3) for k,v in word_to_index.items()}  # Shift indices by 3 to accommodate special tokens
word_to_index["<PAD>"] = 0  # Padding token
word_to_index["<START>"] = 1  # Start-of-sequence token
word_to_index["<UNK>"] = 2  # Unknown word token

# Reverse the word-to-index mapping to index-to-word mapping
index_to_word = {value:key for key,value in word_to_index.items()}

# Function to decode reviews from index representation to words
def decode_review(indexes):
    return ' '.join(index_to_word[index] for index in indexes)

# Decode and print a couple of reviews from the training set
print('Decoded review 1:')
print(decode_review(x_train[0]))  # Decode the first review in the training set
print('Review sentiment:', y_train[0])  # Print the sentiment (0 for negative, 1 for positive)

print('\nDecoded review 2:')
print(decode_review(x_train[2]))  # Decode another review from the training set
print('Review sentiment:', y_train[1])  # Print the sentiment (0 for negative, 1 for positive)


Decoded review 1:
<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for retail and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also congratulations to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the praising list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazi

# 3. Preprocessing Data

Use pad_sequences to ensure all sequences (x_train and x_test) have the same length (maxlen). This is necessary because LSTM networks require inputs to have the same length.

In [11]:
# Pad sequences to ensure uniform length for input to LSTM
print('Pad sequences (samples x time)')
x_train = pad_sequences(x_train, maxlen=maxlen)  # Pad or truncate sequences to a fixed length of maxlen
x_test = pad_sequences(x_test, maxlen=maxlen)    # Pad or truncate sequences in the test set to the same length
print('x_train shape:', x_train.shape)  # Output the shape of the padded training data
print('x_test shape:', x_test.shape)    # Output the shape of the padded test data


Pad sequences (samples x time)
x_train shape: (2000, 200)
x_test shape: (2000, 200)


# 4. Model Creation & Training

Layers:
- Embedding: Converts positive integers (indexes) into dense vectors of fixed size. max_features specifies the size of the vocabulary, embedding_dims is the dimension of the dense embedding, and input_length is the length of input sequences.
- LSTM: Long Short-Term Memory layer with 128 units.
- Dense: Output layer with a single unit and sigmoid activation function for binary classification.

In [12]:
# Define the LSTM model
print('Building model...')
model = Sequential()  # Initialize a sequential model
model.add(Embedding(max_features, embedding_dims, input_length=maxlen))  # Embedding layer: Turns positive integers into dense vectors of fixed size
model.add(LSTM(128))  # LSTM layer with 128 units
model.add(Dense(1, activation='sigmoid'))  # Output layer with sigmoid activation for binary classification

Building model...


In [13]:
# Display the model architecture
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
# Compile the model
model.compile(optimizer='adam',  # Optimizer for gradient descent (Adam)
              loss='binary_crossentropy',  # Loss function for binary classification
              metrics=['accuracy'])  # Metric to monitor during training (accuracy)

print('Training model...')
# Train the model
model.fit(x_train, y_train,  # Training data and labels
          batch_size=batch_size,  # Number of samples per gradient update
          epochs=epochs,  # Number of iterations over the entire dataset
          validation_data=(x_test, y_test))  # Data on which to evaluate the loss and any model metrics at the end of each epoch

Training model...
Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 566ms/step - accuracy: 0.5405 - loss: 0.6869 - val_accuracy: 0.5395 - val_loss: 0.6628
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 557ms/step - accuracy: 0.7398 - loss: 0.5272 - val_accuracy: 0.7800 - val_loss: 0.4755
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 40s 549ms/step - accuracy: 0.9338 - loss: 0.1922 - val_accuracy: 0.7830 - val_loss: 0.5399
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 30s 484ms/step - accuracy: 0.9836 - loss: 0.0611 - val_accuracy: 0.7505 - val_loss: 0.6061
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 34s 542ms/step - accuracy: 0.9951 - loss: 0.0285 - val_accuracy: 0.7595 - val_loss: 0.9414


# 5. Evaluation

Print out the test score (loss value) and accuracy of the model on the test data.

In [15]:
# Evaluate the model
score, acc = model.evaluate(x_test, y_test,
                            batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 103ms/step - accuracy: 0.7556 - loss: 0.9718
Test score: 0.9413973093032837
Test accuracy: 0.7595000267028809
